In [2]:
!pip install -qU transformers accelerate bitsandbytes huggingface_hub tabulate

In [3]:
import os
import gc
import torch

# Prevent VRAM fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

torch.cuda.empty_cache()
gc.collect()

print("GPU Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)} GB")

GPU Available: True
Device Name: Tesla T4
Total VRAM: 14.56 GB


In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "deepseek-ai/deepseek-llm-7b-chat"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
print("Model loaded successfully!")

config.json:   0%|          | 0.00/594 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.28k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/4.61M [00:00<?, ?B/s]

[transformers] `tokenizer_class` is `LlamaTokenizerFast` but `tokenizer.json` is ByteLevel; loading `TokenizersBackend`. Set `tokenizer_class` to `TokenizersBackend` in `tokenizer_config.json`.


pytorch_model.bin.index.json:   0%|          | 0.00/22.5k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model.safetensors.index.json:   0%|          | 0.00/23.6k [00:00<?, ?B/s]

Loading weights:   0%|          | 0/273 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

Model loaded successfully!


In [7]:
import json
import time
import re
from tabulate import tabulate

TESTS = [
    {"dim": "Reasoning", "name": "Train Problem", "prompt": "A train leaves A at 60mph, another leaves B at 80mph toward A. Distance 420mi. When do they meet? Show steps.", "check": lambda r: "3" in r and ("hour" in r.lower() or "h" in r.lower())},
    {"dim": "Reasoning", "name": "Bat & Ball", "prompt": "A bat and a ball cost $1.10 total. The bat costs $1.00 more than the ball. How much is the ball? Show reasoning.", "check": lambda r: "0.05" in r or "5 cent" in r.lower()},
    {"dim": "Coding", "name": "LRU Cache", "prompt": "Write a Python LRU cache class with capacity N, get/put O(1), type hints, docstring.", "check": lambda r: "class" in r and ("OrderedDict" in r or "dict" in r) and "def get" in r and "def put" in r},
    {"dim": "Coding", "name": "Bug Fix", "prompt": "Fix the bug:\ndef find_max(arr):\n    max_val = 0\n    for i in arr:\n        if i > max_val:\n            max_val = i\n    return max_val", "check": lambda r: "max_val = arr[0]" in r or "float('-inf')" in r or "if not arr" in r},
    {"dim": "Coding", "name": "Regex Explain", "prompt": "Explain this regex: r'(?<!\\\\)\\\\.(?=([^\"\\\\]*(\\\\.|\\\"([^\"\\\\]*\\\\.)*[^\"\\\\]*\\\"))*[^\"\\\\]*$)'", "check": lambda r: "escape" in r.lower() or "dot" in r.lower() or "quote" in r.lower()},
    {"dim": "Instruction Following", "name": "JSON Only", "prompt": "Return ONLY valid JSON with keys: name, email, risk_level. No markdown. Data: John Doe, john@co.com, high", "check": lambda r: r.strip().startswith("{") and r.strip().endswith("}") and "name" in r and "email" in r},
    {"dim": "Writing", "name": "CEO Summary", "prompt": "Summarize in 3 bullets for a CEO: 'Transformer models use self-attention to process sequences in parallel. Unlike RNNs, they don't process tokens sequentially, enabling GPU utilization. The attention mechanism computes relationships between all token pairs. Multi-head attention allows capturing different relationship types. Positional encodings inject order information. Layer normalization and residual connections stabilize deep training. Feed-forward networks apply pointwise transformations. This architecture scales effectively to billions of parameters.'", "check": lambda r: r.count("\n") >= 2 or r.count("•") >= 2 or r.count("-") >= 2},
    {"dim": "Writing", "name": "Delay Email", "prompt": "Write a professional email to a client explaining a 2-week project delay due to unexpected technical debt, proposing a revised timeline and mitigation steps.", "check": lambda r: "delay" in r.lower() and ("technical debt" in r.lower() or "revised" in r.lower())},
    {"dim": "Structured Output", "name": "Entity Extraction", "prompt": 'Extract name, email, company, pain_points as JSON: "Hi, I\'m Sarah Chen (sarah@techcorp.io) from TechCorp. We struggle with slow CI/CD and flaky tests."', "check": lambda r: "sarah" in r.lower() and "techcorp" in r.lower() and "ci/cd" in r.lower()},
    {"dim": "Structured Output", "name": "NL to SQL", "prompt": "Generate SQL: 'Show all users who signed up in January 2024 and have made at least 3 orders.' Tables: users(id, created_at), orders(id, user_id, created_at)", "check": lambda r: "SELECT" in r.upper() and "JOIN" in r.upper() and "COUNT" in r.upper()},
    {"dim": "Context", "name": "Long Context QA", "prompt": "Context: The project codename is 'Project Falcon'. The budget is $2.4M. The deadline is Q3 2025.\n\nQuestion: What is the project codename and budget? Answer based ONLY on the text above.", "check": lambda r: "falcon" in r.lower() and "2.4" in r},
    {"dim": "Multilingual", "name": "Japanese Translation", "prompt": "Translate to Japanese: 'Our Q3 revenue increased 12% YoY driven by APAC expansion.'", "check": lambda r: any(c in r for c in "増収益四半期")},
]


results = []
print(f"{'='*60}\nRUNNING {len(TESTS)} TESTS (Clean State)\n{'='*60}")

for i, test in enumerate(TESTS, 1):
    print(f"\n[{i}/{len(TESTS)}] {test['dim']} - {test['name']}")

    messages = [{"role": "user", "content": test["prompt"]}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(model.device)

    start = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=400,  # Slightly capped to save VRAM on long contexts
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.05,
        )
    torch.cuda.synchronize()
    elapsed = time.time() - start

    input_len = inputs["input_ids"].shape[-1]
    new_toks = outputs.shape[1] - input_len

    raw = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)

    passed = test["check"](raw)
    score = 1 if passed else 0
    vram_mb = torch.cuda.memory_allocated() / (1024 ** 2)

    results.append({
        "dimension": test["dim"],
        "test": test["name"],
        "passed": passed,
        "score": score,
        "tok_s": round(new_toks / elapsed, 1),
        "latency_s": round(elapsed, 2),
        "vram_mb": round(vram_mb, 1),
        "response": raw[:300] + "..." if len(raw) > 300 else raw,
    })

    status = "✅ PASS" if passed else "❌ FAIL"
    print(f"  {status} | {round(new_toks / elapsed, 1)} tok/s | {round(elapsed, 2)}s | VRAM: {round(vram_mb, 1)}MB")

    # Aggressively clean GPU cache after each test iteration
    del inputs, outputs, raw
    torch.cuda.empty_cache()
    gc.collect()

# Summary
print(f"\n{'='*60}\nDIMENSION SUMMARY\n{'='*60}")
dim_scores = {}
for r in results:
    dim_scores.setdefault(r["dimension"], {"pass": 0, "total": 0, "tok_s": [], "latency": []})
    dim_scores[r["dimension"]]["pass"] += r["score"]
    dim_scores[r["dimension"]]["total"] += 1
    dim_scores[r["dimension"]]["tok_s"].append(r["tok_s"])
    dim_scores[r["dimension"]]["latency"].append(r["latency_s"])

summary_rows = []
for dim, stats in dim_scores.items():
    pct = (stats["pass"] / stats["total"]) * 100
    avg_tok = sum(stats["tok_s"]) / len(stats["tok_s"])
    avg_lat = sum(stats["latency"]) / len(stats["latency"])
    summary_rows.append([dim, f"{stats['pass']}/{stats['total']}", f"{pct:.0f}%", f"{avg_tok:.1f}", f"{avg_lat:.2f}s"])

print(tabulate(summary_rows, headers=["Dimension", "Passed", "Score", "Avg tok/s", "Avg latency"], tablefmt="github"))

total_pass = sum(r["score"] for r in results)
total_all = len(results)
print(f"\nOVERALL: {total_pass}/{total_all} ({total_pass/total_all*100:.0f}%)")

RUNNING 12 TESTS (Clean State)

[1/12] Reasoning - Train Problem
  ✅ PASS | 13.7 tok/s | 14.05s | VRAM: 4595.9MB

[2/12] Reasoning - Bat & Ball
  ✅ PASS | 14.4 tok/s | 10.38s | VRAM: 4595.9MB

[3/12] Coding - LRU Cache
  ❌ FAIL | 13.9 tok/s | 28.73s | VRAM: 4595.9MB

[4/12] Coding - Bug Fix
  ❌ FAIL | 14.0 tok/s | 9.13s | VRAM: 4595.9MB

[5/12] Coding - Regex Explain
  ✅ PASS | 14.1 tok/s | 28.43s | VRAM: 4595.9MB

[6/12] Instruction Following - JSON Only
  ✅ PASS | 14.5 tok/s | 2.21s | VRAM: 4595.9MB

[7/12] Writing - CEO Summary
  ✅ PASS | 13.8 tok/s | 5.28s | VRAM: 4595.9MB

[8/12] Writing - Delay Email
  ✅ PASS | 14.0 tok/s | 28.49s | VRAM: 4595.9MB

[9/12] Structured Output - Entity Extraction
  ✅ PASS | 14.7 tok/s | 3.4s | VRAM: 4595.9MB

[10/12] Structured Output - NL to SQL
  ✅ PASS | 14.3 tok/s | 14.35s | VRAM: 4595.9MB

[11/12] Context - Long Context QA
  ✅ PASS | 14.5 tok/s | 1.38s | VRAM: 4595.9MB

[12/12] Multilingual - Japanese Translation
  ✅ PASS | 12.4 tok/s | 3.54s | 